# XPOS + UPOS Fine-tuning — XLM-R Large
**Task:** sequence labeling — predict MTE XPOS tag per token; derive UD UPOS from XPOS.  
**Data:** `tor_train/dev/test.tsv` — `form\tlemma\txpos`, blank-line sentence boundaries.  
**Runtime:** L4 GPU (24 GB) recommended.

Upload the `dataset/` folder to `MyDrive/TorlakTag/dataset/` before running.

In [ ]:
!pip install transformers torch seqeval -q

In [ ]:
import os, json, random
import numpy as np
import torch
from google.colab import drive
drive.mount('/content/drive')

# ── CONFIG ────────────────────────────────────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/TorlakTag'
TRAIN_TSV   = f'{DRIVE_ROOT}/dataset/tor_train.tsv'
DEV_TSV     = f'{DRIVE_ROOT}/dataset/tor_dev.tsv'
TEST_TSV    = f'{DRIVE_ROOT}/dataset/tor_test.tsv'
AUG_TRAIN_TSV = f'{DRIVE_ROOT}/dataset/tor_augmented_train.tsv'
AUG_DEV_TSV   = f'{DRIVE_ROOT}/dataset/tor_augmented_dev.tsv'
MTE2UD      = f'{DRIVE_ROOT}/dataset/mte2ud_output.txt'

# Previous best checkpoint to resume from (86.22% run).
# Set to None to train from XLM-R pretrained weights instead.
PREV_CKPT   = f'{DRIVE_ROOT}/models/xlmr_xpos/best_model.pt'

# New save directory — keeps previous run intact
SAVE_DIR    = f'{DRIVE_ROOT}/models/xlmr_xpos_3'

USE_AUGMENTED = True

MODEL_NAME   = 'xlm-roberta-large'
MAX_LEN      = 128
BATCH_SIZE   = 32
WEIGHT_DECAY = 0.01
EPOCHS       = 20    # more headroom — early stopping will cut it short if needed
WARMUP_RATIO = 0.1
SEED         = 42

os.makedirs(SAVE_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os
from collections import Counter

def load_tsv(path):
    """Parse form\tlemma\txpos TSV; blank/tab-only lines = sentence boundaries."""
    sentences, current = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if current:
                    sentences.append(current)
                    current = []
            else:
                parts = line.split('\t')
                if len(parts) == 3 and parts[0].strip():
                    current.append({'form': parts[0], 'lemma': parts[1], 'xpos': parts[2]})
    if current:
        sentences.append(current)
    return sentences

train_sents    = load_tsv(TRAIN_TSV)
orig_dev_sents = load_tsv(DEV_TSV)   # kept pure — used for early-stopping signal
dev_sents      = list(orig_dev_sents)
test_sents     = load_tsv(TEST_TSV)

# Merge augmented data when available
if USE_AUGMENTED:
    for aug_path, target in [(AUG_TRAIN_TSV, 'train'), (AUG_DEV_TSV, 'dev')]:
        if os.path.exists(aug_path):
            aug = load_tsv(aug_path)
            if target == 'train':
                train_sents = train_sents + aug
                print(f'Merged {len(aug)} augmented train sentences')
            else:
                dev_sents = dev_sents + aug  # used for loss only, NOT for checkpointing
                print(f'Merged {len(aug)} augmented dev sentences')
        else:
            print(f'WARNING: {aug_path} not found — run augment_xpos.ipynb first')

for split, sents in [('train', train_sents), ('dev (aug+orig)', dev_sents),
                     ('dev (orig only)', orig_dev_sents), ('test', test_sents)]:
    toks = sum(len(s) for s in sents)
    print(f'{split:20s}: {len(sents):5d} sentences  {toks:6d} tokens')

In [ ]:
from collections import Counter

# ── Load MTE → UD mapping ─────────────────────────────────────────────────────
def load_mte2ud(path):
    """
    Parse mte2ud_output.txt (3-column TSV: mte_tag, upos, feats).
    Returns two dicts:
      xpos2upos  : mte_tag -> UD UPOS string
      xpos2feats : mte_tag -> UD morphological features string (or '_')
    Malformed lines (MTE tag contains a space, or < 3 columns) are skipped.
    """
    xpos2upos, xpos2feats = {}, {}
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                continue
            parts = line.split('\t')
            mte   = parts[0].strip()
            if ' ' in mte or len(parts) < 3:   # skip 2 known malformed lines
                continue
            xpos2upos[mte]  = parts[1].strip()
            xpos2feats[mte] = parts[2].strip()
    return xpos2upos, xpos2feats

xpos2upos, xpos2feats = load_mte2ud(MTE2UD)
print(f'MTE2UD loaded: {len(xpos2upos)} entries')

# Lookup functions — used at inference and in the evaluate breakdown table
def xpos_to_upos(xpos):
    """Exact lookup; falls back to first-char heuristic for unseen tags."""
    if xpos in xpos2upos:
        return xpos2upos[xpos]
    # fallback for tags not in the mapping file
    c, sub = xpos[0].upper() if xpos else 'X', xpos[1] if len(xpos) > 1 else ''
    return {
        'N': 'PROPN' if sub == 'p' else 'NOUN',
        'V': 'AUX'   if sub in ('a', 'l', 'c') else 'VERB',
        'A': 'ADJ', 'P': 'PRON', 'M': 'NUM', 'R': 'ADV',
        'S': 'ADP',  'C': 'SCONJ' if sub == 's' else 'CCONJ',
        'Q': 'PART', 'I': 'INTJ', 'Z': 'PUNCT', 'X': 'X',
    }.get(c, 'X')

def xpos_to_feats(xpos):
    """Return UD morphological features string for a given XPOS tag."""
    return xpos2feats.get(xpos, '_')

# ── Build XPOS label vocabulary from training data ────────────────────────────
xpos_counts = Counter(tok['xpos'] for sent in train_sents for tok in sent)
id2xpos = ['[UNK]'] + sorted(xpos_counts)
xpos2id = {tag: i for i, tag in enumerate(id2xpos)}
NUM_LABELS = len(id2xpos)

print(f'XPOS vocab size : {NUM_LABELS}  (+ [UNK] for tags unseen in train)')
print(f'Top-10 tags     : {xpos_counts.most_common(10)}')

# Quick sanity check: key edge cases the old heuristic got wrong
for tag, expected_upos in [('Mdo','ADJ'), ('Ps1fsn','DET'), ('Xa','SYM'), ('Vcr3s','AUX')]:
    got = xpos_to_upos(tag)
    status = '✓' if got == expected_upos else f'✗ (got {got})'
    print(f'  {tag} -> {got}  {status}')

In [ ]:
# ── Morphological feature extraction for auxiliary multi-task loss ─────────────
# MTE BCS tags encode features positionally.  We extract 4 key features and add
# lightweight auxiliary heads that share the XLM-R encoder.
# This forces the model to learn animate/inanimate, gender, number, and case
# as explicit signals — directly fixing the Agpfsn vs Agpfsny confusion.

ANIMATE_CLASSES = 2   # 0 = inanimate/NA,  1 = animate
GENDER_CLASSES  = 4   # 0 = m,  1 = f,  2 = n,  3 = other/NA
NUMBER_CLASSES  = 3   # 0 = singular,  1 = plural,  2 = other/NA
CASE_CLASSES    = 8   # 0=n 1=g 2=d 3=a 4=l 5=i 6=v 7=other

def extract_morph_features(xpos: str):
    """
    Return (animate, gender, number, case) class indices for one MTE XPOS tag.
    Works for N/A/P classes; verbs and invariable tags get 'other' values.
    """
    # Strip toponym/dialect suffixes so 'y' detection is clean
    stripped = xpos.replace('-t','').replace('-n','').replace('-v','').replace('-','')

    # Animacy: 'y' in stripped tag = animate form
    animate = 1 if 'y' in stripped else 0

    # Gender: first m/f/n at position >= 2
    gender = 3
    for c in (xpos[2:6] if len(xpos) > 2 else ''):
        if c == 'm': gender = 0; break
        if c == 'f': gender = 1; break
        if c == 'n': gender = 2; break

    # Number: first s/p at position >= 3
    number = 2
    for c in (xpos[3:7] if len(xpos) > 3 else ''):
        if c == 's': number = 0; break
        if c == 'p': number = 1; break

    # Case: first case letter at position >= 4
    case_map = {'n': 0, 'g': 1, 'd': 2, 'a': 3, 'l': 4, 'i': 5, 'v': 6}
    case = 7
    for c in (xpos[4:] if len(xpos) > 4 else ''):
        if c in case_map: case = case_map[c]; break

    return animate, gender, number, case

# Sanity check on the key confused tag pairs
print(f'{"Tag":<14} {"animate":<12} {"gender":<8} {"number":<8} case')
print('-' * 50)
for tag in ['Agpfsn', 'Agpfsny', 'Agpmpn', 'Agpmpny', 'Agpfpny', 'Ncmsn', 'Vcp-sn']:
    a, g, n, c = extract_morph_features(tag)
    print(f'{tag:<14} {"animate" if a else "inanimate":<12} '
          f'{["m","f","n","?"][g]:<8} {["sg","pl","?"][n]:<8} '
          f'{["n","g","d","a","l","i","v","?"][c]}')

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(sentences, max_len=MAX_LEN):
    """
    Tokenize sentences with XLM-R.
    Returns a dict of tensors — main labels + auxiliary morphological feature labels.
    All continuation subwords and special tokens get -100 (ignored by loss).
    """
    all_ids, all_mask = [], []
    all_labels, all_animate, all_gender, all_number, all_case = [], [], [], [], []
    unk = xpos2id['[UNK]']

    for sent in sentences:
        words = [t['form'] for t in sent]
        tags  = [xpos2id.get(t['xpos'], unk) for t in sent]
        feats = [extract_morph_features(t['xpos']) for t in sent]

        enc      = tokenizer(words, is_split_into_words=True,
                             max_length=max_len, truncation=True,
                             padding='max_length')
        word_ids = enc.word_ids()

        labels, animate_l, gender_l, number_l, case_l = [], [], [], [], []
        prev = None
        for wid in word_ids:
            if wid is None or wid == prev:
                labels.append(-100)
                animate_l.append(-100); gender_l.append(-100)
                number_l.append(-100);  case_l.append(-100)
            else:
                labels.append(tags[wid] if wid < len(tags) else -100)
                if wid < len(feats):
                    a, g, n, c = feats[wid]
                    animate_l.append(a); gender_l.append(g)
                    number_l.append(n);  case_l.append(c)
                else:
                    animate_l.append(-100); gender_l.append(-100)
                    number_l.append(-100);  case_l.append(-100)
            prev = wid

        all_ids.append(enc['input_ids'])
        all_mask.append(enc['attention_mask'])
        all_labels.append(labels)
        all_animate.append(animate_l); all_gender.append(gender_l)
        all_number.append(number_l);   all_case.append(case_l)

    return {
        'input_ids':      torch.tensor(all_ids,     dtype=torch.long),
        'attention_mask': torch.tensor(all_mask,    dtype=torch.long),
        'labels':         torch.tensor(all_labels,  dtype=torch.long),
        'animate':        torch.tensor(all_animate, dtype=torch.long),
        'gender':         torch.tensor(all_gender,  dtype=torch.long),
        'number':         torch.tensor(all_number,  dtype=torch.long),
        'case':           torch.tensor(all_case,    dtype=torch.long),
    }

print('Encoding …')
train_enc    = encode(train_sents)
dev_enc      = encode(dev_sents)
orig_dev_enc = encode(orig_dev_sents)   # clean dev — used for early stopping
test_enc     = encode(test_sents)
print(f'Train shape: {train_enc["input_ids"].shape}')

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MorphoDataset(Dataset):
    def __init__(self, enc):
        self.enc = enc
    def __len__(self):
        return len(self.enc['input_ids'])
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}

train_loader    = DataLoader(MorphoDataset(train_enc),    batch_size=BATCH_SIZE,
                             shuffle=True,  num_workers=0, pin_memory=True)
dev_loader      = DataLoader(MorphoDataset(dev_enc),      batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=0, pin_memory=True)
orig_dev_loader = DataLoader(MorphoDataset(orig_dev_enc), batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=0, pin_memory=True)
test_loader     = DataLoader(MorphoDataset(test_enc),     batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=0, pin_memory=True)
print(f'Batches — train: {len(train_loader)}  dev: {len(dev_loader)}  '
      f'orig_dev: {len(orig_dev_loader)}  test: {len(test_loader)}')

In [ ]:
import torch.nn as nn
from transformers import AutoModel

class XLMRForXPOS(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        h               = self.encoder.config.hidden_size
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(h, num_labels)
        self.animate_head = nn.Linear(h, ANIMATE_CLASSES)
        self.gender_head  = nn.Linear(h, GENDER_CLASSES)
        self.number_head  = nn.Linear(h, NUMBER_CLASSES)
        self.case_head    = nn.Linear(h, CASE_CLASSES)

    def forward(self, input_ids, attention_mask,
                labels=None, animate=None, gender=None, number=None, case=None,
                **kwargs):
        seq    = self.encoder(input_ids, attention_mask=attention_mask).last_hidden_state
        h      = self.dropout(seq)
        logits = self.classifier(h)
        loss   = None
        if labels is not None:
            ce = nn.CrossEntropyLoss(ignore_index=-100)
            xpos_loss = ce(logits.view(-1, NUM_LABELS), labels.view(-1))
            aux = (
                ce(self.animate_head(h).view(-1, ANIMATE_CLASSES), animate.view(-1)) * 0.4
              + ce(self.gender_head(h).view(-1,  GENDER_CLASSES),  gender.view(-1))  * 0.2
              + ce(self.number_head(h).view(-1,  NUMBER_CLASSES),  number.view(-1))  * 0.2
              + ce(self.case_head(h).view(-1,    CASE_CLASSES),    case.view(-1))    * 0.2
            )
            loss = xpos_loss + 0.25 * aux
        return logits, loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = XLMRForXPOS(MODEL_NAME, NUM_LABELS).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# ── Load previous best checkpoint ─────────────────────────────────────────────
# PREV_CKPT points to the 86.22% model; SAVE_DIR is a fresh directory.
# strict=False: aux heads not in the old checkpoint → randomly initialized (expected).
ckpt_path = f'{SAVE_DIR}/best_model.pt'
if PREV_CKPT and os.path.exists(PREV_CKPT):
    ckpt = torch.load(PREV_CKPT, map_location=device)
    saved_labels = ckpt['classifier.weight'].shape[0]
    info = model.load_state_dict(ckpt, strict=False)
    if saved_labels != NUM_LABELS:
        nn.init.xavier_uniform_(model.classifier.weight)
        nn.init.zeros_(model.classifier.bias)
        print(f'✓ Encoder loaded from {PREV_CKPT}')
        print(f'  Classifier reinitialized: {saved_labels}→{NUM_LABELS} labels')
    else:
        print(f'✓ Encoder + classifier loaded from {PREV_CKPT}  ({NUM_LABELS} labels)')
    aux_keys = [k for k in info.missing_keys if 'head' in k]
    if aux_keys:
        print(f'  Aux heads randomly initialized ({len(aux_keys)} tensors — expected)')
elif os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'✓ Resumed from existing SAVE_DIR checkpoint')
else:
    print('No checkpoint found — training from XLM-R pretrained weights')

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm.auto import tqdm

# ── Class weights on primary XPOS loss ────────────────────────────────────────
tag_freq = Counter(tok['xpos'] for sent in train_sents for tok in sent)
weights  = np.zeros(NUM_LABELS, dtype=np.float32)
for tag, idx in xpos2id.items():
    weights[idx] = 0.0 if tag == '[UNK]' else 1.0 / np.sqrt(max(tag_freq.get(tag, 1), 1))
class_weights = torch.tensor(weights).to(device)

def weighted_forward(batch):
    ids  = batch['input_ids'].to(device)
    mask = batch['attention_mask'].to(device)
    lbls = batch['labels'].to(device)
    ani  = batch['animate'].to(device)
    gen  = batch['gender'].to(device)
    num  = batch['number'].to(device)
    cas  = batch['case'].to(device)

    seq    = model.encoder(ids, attention_mask=mask).last_hidden_state
    h      = model.dropout(seq)
    logits = model.classifier(h)

    ce_w      = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
    ce        = nn.CrossEntropyLoss(ignore_index=-100)
    xpos_loss = ce_w(logits.view(-1, NUM_LABELS), lbls.view(-1))
    aux = (
        ce(model.animate_head(h).view(-1, ANIMATE_CLASSES), ani.view(-1)) * 0.4
      + ce(model.gender_head(h).view(-1,  GENDER_CLASSES),  gen.view(-1)) * 0.2
      + ce(model.number_head(h).view(-1,  NUMBER_CLASSES),  num.view(-1)) * 0.2
      + ce(model.case_head(h).view(-1,    CASE_CLASSES),    cas.view(-1)) * 0.2
    )
    return logits, xpos_loss + 0.25 * aux

# ── Optimizer: lower LR on encoder since it's already task-adapted ─────────────
# Encoder at 5e-6 (half of before) — fine-tune the already good 86% model gently.
# Heads at 3e-5 — they're new/untrained and need faster updates.
optimizer = AdamW([
    {'params': model.encoder.parameters(),                              'lr': 5e-6},
    {'params': list(model.classifier.parameters()) +
               list(model.animate_head.parameters()) +
               list(model.gender_head.parameters()) +
               list(model.number_head.parameters()) +
               list(model.case_head.parameters()),                      'lr': 3e-5},
], weight_decay=WEIGHT_DECAY)

# T_0=5 with eta_min=5e-8: gentler lower bound than before
PATIENCE  = 5
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=1, eta_min=5e-8)

def evaluate(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            logits, _ = model(ids, mask)
            preds = logits.argmax(-1)
            valid = lbls != -100
            correct += (preds[valid] == lbls[valid]).sum().item()
            total   += valid.sum().item()
    return correct / total if total else 0.0

best_dev   = -1.0
no_improve = 0
ckpt_path  = f'{SAVE_DIR}/best_model.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'):
        logits, loss = weighted_forward(batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(epoch - 1); optimizer.zero_grad()
        running += loss.item()

    # Checkpoint on ORIGINAL dev only — clean signal, not polluted by augmented dev
    orig_acc = evaluate(orig_dev_loader)
    enc_lr   = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:2d}  loss={running/len(train_loader):.4f}  '
          f'orig_dev={orig_acc:.4f}  lr(enc)={enc_lr:.1e}')

    if orig_acc > best_dev:
        best_dev   = orig_acc
        no_improve = 0
        torch.save(model.state_dict(), ckpt_path)
        print(f'  ✓ Saved  (best orig_dev: {best_dev:.4f})')
    else:
        no_improve += 1
        print(f'  No improvement ({no_improve}/{PATIENCE})')
        if no_improve >= PATIENCE:
            print('Early stopping triggered.')
            break

In [ ]:
from collections import defaultdict

# Load best checkpoint
model.load_state_dict(torch.load(f'{SAVE_DIR}/best_model.pt', map_location=device))
test_acc = evaluate(test_loader)
print(f'Test XPOS accuracy: {test_acc:.4f}')

# Per-class breakdown
model.eval()
cls_correct = defaultdict(int)
cls_total   = defaultdict(int)
with torch.no_grad():
    for batch in test_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        preds = model(ids, mask)[0].argmax(-1)
        for p, g in zip(preds.view(-1).tolist(), lbls.view(-1).tolist()):
            if g == -100:
                continue
            tag = id2xpos[g]
            cls_correct[tag] += int(p == g)
            cls_total[tag]   += 1

print('\nWorst-performing XPOS tags (>= 5 test instances):')
rows = [(t, cls_correct[t] / cls_total[t], cls_total[t])
        for t in cls_total if cls_total[t] >= 5]
rows.sort(key=lambda x: x[1])
print(f'  {"XPOS":<18} {"UPOS":<8} {"acc":>6}  n')
for tag, acc, n in rows[:25]:
    print(f'  {tag:<18} {xpos_to_upos(tag):<8} {acc:>6.2%}  {n}')

# Save label vocab
with open(f'{SAVE_DIR}/xpos_vocab.json', 'w', encoding='utf-8') as f:
    json.dump({'id2xpos': id2xpos, 'xpos2id': xpos2id}, f, ensure_ascii=False)
print(f'\nVocab + model saved to {SAVE_DIR}')

In [ ]:
def predict(words):
    """
    Predict XPOS, UPOS, and UD morphological features for a list of word forms.
    UPOS and FEATS are derived from the predicted XPOS via the MTE2UD lookup —
    no extra model needed, and UPOS/XPOS are always consistent.
    """
    model.eval()
    enc      = tokenizer(words, is_split_into_words=True,
                         max_length=MAX_LEN, truncation=True,
                         return_tensors='pt')
    word_ids = enc.word_ids()   # must be called on BatchEncoding before .to(device)
    enc      = enc.to(device)
    with torch.no_grad():
        logits, _ = model(**enc)

    seen, results = set(), []
    for i, wid in enumerate(word_ids):
        if wid is not None and wid not in seen:
            xpos  = id2xpos[logits[0, i].argmax().item()]
            upos  = xpos_to_upos(xpos)
            feats = xpos_to_feats(xpos)
            results.append((words[wid], xpos, upos, feats))
            seen.add(wid)
    return results

# Example from the Torlak transcript
example = ['i', 'on', 'pade', 'i', 'tolko', 'se', 'ubije', 'mnogo']
print(f'{"form":<18} {"XPOS":<18} {"UPOS":<8} FEATS')
print('-' * 80)
for form, xpos, upos, feats in predict(example):
    print(f'{form:<18} {xpos:<18} {upos:<8} {feats}')